## Exercise: Text Preprocessing with Bible Text

For this exercise, you will import text from https://raw.githubusercontent.com/mxw/grmr/refs/heads/master/src/finaltests/bible.txt and apply the text preprocessing techniques we've learned using Keras Tokenizer and `pad_sequences`.

### Task 1: Fetching the Text Data

In [1]:
import requests

# URL of the text file
url = "https://raw.githubusercontent.com/mxw/grmr/refs/heads/master/src/finaltests/bible.txt"

# Fetch the content
response = requests.get(url)
bible_text = response.text

# Display the first 500 characters to verify
print(bible_text[:500])

1:1 In the beginning God created the heaven and the earth.

1:2 And the earth was without form, and void; and darkness was upon
the face of the deep. And the Spirit of God moved upon the face of the
waters.

1:3 And God said, Let there be light: and there was light.

1:4 And God saw the light, that it was good: and God divided the light
from the darkness.

1:5 And God called the light Day, and the darkness he called Night.
And the evening and the morning were the first day.

1:6 An


### Task 2: Preprocessing the Text

Apply tokenization, convert to sequences, and pad the sequences. Consider how you might split the large text into manageable 'sentences' or paragraphs for tokenization.

In [2]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Split the text into lines/paragraphs as a proxy for sentences
# Filter out empty lines
bible_sentences = [line.strip() for line in bible_text.split('\n') if line.strip()]

# Initialize Tokenizer
# Consider a reasonable num_words and oov_token
# For this large text, you might want to increase num_words significantly
# Or, set it to None to tokenize all words, but be mindful of memory usage.
tokenizer_bible = Tokenizer(num_words=10000, oov_token='<unk>')

# Fit on the bible_sentences
tokenizer_bible.fit_on_texts(bible_sentences)

# Display some info about the tokenizer
print(f"Total unique words found: {len(tokenizer_bible.word_index)}")
print("Top 10 words and their indices:")
for word, index in list(tokenizer_bible.word_index.items())[:10]:
    print(f"  {word}: {index}")

# Convert text to sequences
bible_sequences = tokenizer_bible.texts_to_sequences(bible_sentences)

# Determine an appropriate maxlen for padding
# A common approach is to use a percentile of sentence lengths
sequence_lengths = [len(seq) for seq in bible_sequences if len(seq) > 0]
if sequence_lengths:
    median_length = int(np.median(sequence_lengths))
    max_sequence_length = min(median_length * 2, 200) # Cap at 200 or 2x median, adjust as needed
else:
    max_sequence_length = 50 # Default if no sequences

print(f"\nMedian sequence length: {median_length if sequence_lengths else 'N/A'}")
print(f"Chosen maxlen for padding: {max_sequence_length}")

# Pad the sequences
padded_bible_sequences = pad_sequences(bible_sequences,
                                         maxlen=max_sequence_length,
                                         padding='post',
                                         truncating='post')

# Display the first few padded sequences
print("\nFirst 5 padded sequences (truncated to 10 elements for display):")
for i, seq in enumerate(padded_bible_sequences[:5]):
    print(f"  Sequence {i+1}: {seq[:10]}...")

Total unique words found: 13367
Top 10 words and their indices:
  <unk>: 1
  the: 2
  and: 3
  of: 4
  to: 5
  that: 6
  in: 7
  he: 8
  shall: 9
  unto: 10

Median sequence length: 13
Chosen maxlen for padding: 26

First 5 padded sequences (truncated to 10 elements for display):
  Sequence 1: [  43   43    7    2  727   28 1336    2  204    3]...
  Sequence 2: [  43   48    3    2  137   27  257 1918    3 2050]...
  Sequence 3: [   2  264    4    2 1036    3    2  223    4   28]...
  Sequence 4: [343   0   0   0   0   0   0   0   0   0]...
  Sequence 5: [ 43  52   3  28  32  98  59  17 368   3]...


### Task 3: Critical Thinking Questions

Answer the following questions based on your preprocessing steps:

1.  **Tokenizer Parameters:** How did your choice of `num_words` and `oov_token` affect the `word_index` and subsequent `bible_sequences`? What are the trade-offs of setting `num_words` to a very high value versus a lower one?
2.  **Sequence Length and Padding:** How did you determine the `maxlen` for `pad_sequences`? What happens if `maxlen` is too small or too large? Describe the impact of `padding='pre'` vs `padding='post'` and `truncating='pre'` vs `truncating='post'` in the context of sequence data for machine learning models.
3.  **Impact of Standardization:** The default `standardize="lower_and_strip_punctuation"` was used implicitly. How might different standardization rules (e.g., keeping punctuation, not converting to lowercase) affect the tokenization process and the resulting `word_index`? Provide an example of a scenario where you might want to customize standardization.
4.  **Beyond Words:** The `Tokenizer` defaults to word-level tokenization. When might character-level tokenization or n-gram tokenization be more appropriate? What are the challenges and benefits of each approach?